In [11]:
# Imports and config

import os, json, torch, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
import torchvision.models as models
import skimage.draw

# ── Device Setup (Multi-GPU) ───────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_GPUS    = torch.cuda.device_count()
print(f"Device   : {DEVICE}")
print(f"Num GPUs : {NUM_GPUS}")

# ── Label Mapping (background = 0) ────────────────────────────
TOP5_CATEGORIES = {1: 1, 8: 2, 7: 3, 2: 4, 9: 5}
IDX_TO_NAME     = {
    0: 'background',
    1: 'short_sleeve_top',
    2: 'trousers',
    3: 'shorts',
    4: 'long_sleeve_top',
    5: 'skirt'
}
NUM_CLASSES = 6  # 0=background + 5 foreground

# ── Paths ──────────────────────────────────────────────────────
DATA_ROOT = '/kaggle/input/datasets/varun000reddy/'
TRAIN_IMG = DATA_ROOT + 'training/train/image/'
TRAIN_ANN = DATA_ROOT + 'training/train/annos/'
VAL_IMG   = DATA_ROOT + 'validation/validation/image/'
VAL_ANN   = DATA_ROOT + 'validation/validation/annos/'

SAVE_DIR  = '/kaggle/working/checkpoints/'
IMG_SIZE  = 512
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Training Config ────────────────────────────────────────────
# With 2 GPUs + DataParallel:
# batch_size=32, 25k samples, 30 epochs
BATCH_SIZE      = 32
MAX_TRAIN       = 25000
NUM_EPOCHS      = 30
VAL_SUBSET_SIZE = 10000   # used during training
VAL_FREQ        = 1

print(f"Image size     : {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Train samples  : {MAX_TRAIN}")
print(f"Epochs         : {NUM_EPOCHS}")
print(f"Val subset     : {VAL_SUBSET_SIZE}")
print(f"Val frequency  : every {VAL_FREQ} epochs")

Device   : cuda
Num GPUs : 2
Image size     : 512x512
Batch size     : 32
Train samples  : 25000
Epochs         : 30
Val subset     : 10000
Val frequency  : every 1 epochs


In [12]:
# Dataset class

class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, valid_ids_path=None, max_samples=None):
        self.img_dir  = img_dir
        self.mask_dir = mask_dir

        if valid_ids_path and os.path.exists(valid_ids_path):
            with open(valid_ids_path) as f:
                ids = json.load(f)
        else:
            ids = sorted([f.replace('.jpg','') for f in os.listdir(img_dir)])

        if max_samples:
            ids = ids[:max_samples]

        self.ids = ids

        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])

        print(f"Dataset loaded: {len(self.ids)}")

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]

        img  = Image.open(os.path.join(self.img_dir, img_id + '.jpg')).convert('RGB')
        mask = Image.open(os.path.join(self.mask_dir, img_id + '.png'))

        img  = self.transform(img)
        mask = torch.tensor(np.array(mask), dtype=torch.long)

        return img, mask

In [13]:
# Dataset Creation

#  Training Dataset 
TRAIN_ROOT = '/kaggle/input/datasets/pankajdeopaiiitb/vr-processed-segmentation-dataset/'

train_dataset = SegDataset(
    TRAIN_ROOT + 'images',
    TRAIN_ROOT + 'masks',
    TRAIN_ROOT + 'valid_ids.json',
    max_samples=MAX_TRAIN
)

#  Validation Dataset 
VAL_ROOT = '/kaggle/input/datasets/pankajdeopaiiitb/vr-processed-validation-segmentation/'

full_val_dataset = SegDataset(
    VAL_ROOT + 'images',
    VAL_ROOT + 'masks'
)

val_indices  = np.random.choice(len(full_val_dataset),
                                size=VAL_SUBSET_SIZE, replace=False)
val_subset   = Subset(full_val_dataset, val_indices)

print(f"Train       : {len(train_dataset)}")
print(f"Val subset  : {len(val_subset)}")
print(f"Full val    : {len(full_val_dataset)}")

Dataset loaded: 25000
Dataset loaded: 32153
Train       : 25000
Val subset  : 10000
Full val    : 32153


In [5]:
# Data Loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=2
)

full_val_loader = DataLoader(
    full_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    prefetch_factor=2
)

In [14]:
# Model

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)


class UNet(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        resnet        = models.resnet34(weights='IMAGENET1K_V1')
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool     = resnet.maxpool
        self.encoder2 = resnet.layer1
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4

        self.up5  = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec5 = DoubleConv(256 + 256, 256)

        self.up4  = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec4 = DoubleConv(128 + 128, 128)

        self.up3  = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec3 = DoubleConv(64 + 64, 64)

        self.up2  = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec2 = DoubleConv(32 + 64, 32)

        self.up1  = nn.ConvTranspose2d(32, 16, 2, 2)
        self.dec1 = DoubleConv(16, 16)

        self.final = nn.Conv2d(16, num_classes, 1)

    def forward(self, x):
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool(e1))
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        e5 = self.encoder5(e4)

        d5 = self.dec5(torch.cat([self.up5(e5), e4], dim=1))
        d4 = self.dec4(torch.cat([self.up4(d5), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        d1 = self.dec1(self.up1(d2))

        return self.final(d1)


# ── Build model with DataParallel ──────────────────────────────
model = UNet(num_classes=NUM_CLASSES).to(DEVICE)
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel across {NUM_GPUS} GPUs")

total_params = sum(p.numel() for p in model.parameters())
print(f"U-Net ResNet-34 | Parameters: {total_params/1e6:.2f}M")

Using DataParallel across 2 GPUs
U-Net ResNet-34 | Parameters: 24.35M


In [15]:
# Loss, Optimizer & Scheduler

class DiceCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        # CE loss
        ce_loss = self.ce(logits, targets)

        # Dice loss per class
        probs       = torch.softmax(logits, dim=1)
        smooth      = 1e-6
        dice_losses = []
        for cls in range(NUM_CLASSES):
            pred_cls   = probs[:, cls]
            target_cls = (targets == cls).float()
            inter      = (pred_cls * target_cls).sum()
            dice       = 1 - (2 * inter + smooth) / (
                pred_cls.sum() + target_cls.sum() + smooth)
            dice_losses.append(dice)

        return 0.4 * ce_loss + 0.6 * torch.stack(dice_losses).mean()


criterion = DiceCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS)

print("Loss     : DiceCELoss (0.4 CE + 0.6 Dice)")
print("Optimizer: AdamW (lr=3e-4, weight_decay=1e-4)")
print("Scheduler: CosineAnnealingLR")

Loss     : DiceCELoss (0.4 CE + 0.6 Dice)
Optimizer: AdamW (lr=3e-4, weight_decay=1e-4)
Scheduler: CosineAnnealingLR


In [11]:
# Training Loop 

from tqdm import tqdm
import subprocess

def print_gpu_usage():
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
             '--format=csv,nounits,noheader'],
            capture_output=True, text=True
        )
        print("\nGPU Usage:")
        print(result.stdout)
    except:
        print("GPU stats unavailable")


def compute_metrics(logits, targets, num_classes=NUM_CLASSES):
    preds  = logits.argmax(dim=1)
    smooth = 1e-6
    ious, dices = [], []

    for cls in range(1, num_classes):
        pred_cls   = (preds == cls).float()
        target_cls = (targets == cls).float()

        inter = (pred_cls * target_cls).sum().item()
        union = (pred_cls + target_cls).clamp(0, 1).sum().item()

        iou  = (inter + smooth) / (union + smooth)
        dice = (2 * inter + smooth) / (
            pred_cls.sum().item() + target_cls.sum().item() + smooth
        )

        ious.append(iou)
        dices.append(dice)

    return np.mean(ious), np.mean(dices)


def get_state_dict(model):
    if isinstance(model, nn.DataParallel):
        return model.module.state_dict()
    return model.state_dict()


# Upload function to save the checkpoints
def upload_checkpoint_to_kaggle(checkpoint_path, epoch,
                               dataset_id='pankajdeopaiiitb/vr-unet-checkpoint'):
    import subprocess, json, shutil, os

    upload_dir = '/kaggle/working/checkpoint_upload/'
    os.makedirs(upload_dir, exist_ok=True)

    filename = f'unet_epoch_{epoch}.pt'
    shutil.copy(checkpoint_path, upload_dir + filename)

    metadata = {
        "title"   : "vr-unet-checkpoint",
        "id"      : dataset_id,
        "licenses": [{"name": "CC0-1.0"}]
    }

    with open(upload_dir + 'dataset-metadata.json', 'w') as f:
        json.dump(metadata, f)

    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', upload_dir,
         '-m', f"checkpoint epoch {epoch}", '--dir-mode', 'zip'],
        capture_output=True, text=True
    )

    if result.returncode != 0:
        subprocess.run(
            ['kaggle', 'datasets', 'create', '-p', upload_dir,
             '--dir-mode', 'zip'],
            capture_output=True, text=True
        )

    print(f"  ✓ Checkpoint uploaded (epoch {epoch})")


# ── Resume ───────────────────────────────────────────
START_EPOCH = 1
best_miou   = 0.0
history     = []

resume_path = SAVE_DIR + 'unet_latest.pt'
if os.path.exists(resume_path):
    print("Found checkpoint — resuming...")
    ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(ckpt['model_state_dict'])
    else:
        model.load_state_dict(ckpt['model_state_dict'])

    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])

    START_EPOCH = ckpt['epoch'] + 1
    best_miou   = ckpt['best_miou']
    history     = ckpt.get('history', [])

    print(f"Resumed from epoch {ckpt['epoch']} | Best mIoU: {best_miou:.4f}")
else:
    print("No checkpoint — starting fresh")


# ── AMP SCALER ───────────────────────────────
scaler = torch.cuda.amp.GradScaler()


# ── Training Loop ────────────────────────────────────
for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    train_bar = tqdm(train_loader,
                     desc=f"Epoch {epoch:02d}/{NUM_EPOCHS} [Train]")

    for imgs, masks in train_bar:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

        if train_bar.n % 500 == 0:
            print_gpu_usage()

    epoch_loss /= len(train_loader)
    scheduler.step()

    # ── Validation ───────────────────────────────
    val_miou, val_dice = 0.0, 0.0

    if epoch % VAL_FREQ == 0:
        model.eval()
        val_loss = 0.0
        batch_ious, batch_dices = [], []

        with torch.no_grad():
            for imgs, masks in tqdm(val_loader,
                                    desc='Validating', leave=False):
                imgs  = imgs.to(DEVICE, non_blocking=True)
                masks = masks.to(DEVICE, non_blocking=True)

                with torch.cuda.amp.autocast():
                    logits = model(imgs)
                    loss   = criterion(logits, masks)

                miou, dice = compute_metrics(logits, masks)

                val_loss += loss.item()
                batch_ious.append(miou)
                batch_dices.append(dice)

        val_loss /= len(val_loader)
        val_miou  = np.mean(batch_ious)
        val_dice  = np.mean(batch_dices)

        del batch_ious, batch_dices
        torch.cuda.empty_cache()

        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
              f"Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"mIoU: {val_miou:.4f} | Dice: {val_dice:.4f}")

        if val_miou > best_miou:
            best_miou = val_miou
            torch.save(get_state_dict(model), SAVE_DIR + 'unet_best.pt')
            print(f"  ✓ Best model saved (mIoU: {best_miou:.4f})")

    else:
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
              f"Train Loss: {epoch_loss:.4f} | (no val this epoch)")

    history.append({
        'epoch'      : epoch,
        'train_loss' : float(epoch_loss),
        'val_miou'   : float(val_miou),
        'val_dice'   : float(val_dice)
    })

    # Save latest checkpoint
    torch.save({
        'epoch'               : epoch,
        'model_state_dict'    : get_state_dict(model),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_miou'           : best_miou,
        'history'             : history
    }, SAVE_DIR + 'unet_latest.pt')

    print(f"  ✓ Checkpoint saved (epoch {epoch})")

    # Upload every 3 epochs
    if epoch % 3 == 0:
        upload_checkpoint_to_kaggle(
            SAVE_DIR + 'unet_latest.pt',
            epoch
        )

    torch.cuda.empty_cache()
    import gc
    gc.collect()
    
    # GPU usage
    print_gpu_usage()

print("\nTraining complete!")

Found checkpoint — resuming...
Resumed from epoch 15 | Best mIoU: 0.4921


Epoch 16/30 [Train]:   0%|          | 1/782 [00:07<1:38:55,  7.60s/it, loss=0.0884]


GPU Usage:
60, 5615
5, 3973



Epoch 16/30 [Train]:   7%|▋         | 56/782 [00:33<07:17,  1.66it/s, loss=0.0713] 


KeyboardInterrupt: 

In [28]:
MODEL_PATH = '/kaggle/input/datasets/pankajdeopaiiitb/unet-checkpoints-v16/checkpoints/unet_best.pt'

state_dict = torch.load(MODEL_PATH, map_location=DEVICE)

if isinstance(model, nn.DataParallel):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()
print(" Model loaded for validation")

 Model loaded for validation


In [29]:
# Validation on Full Dataset

MODEL_PATH = '/kaggle/input/datasets/pankajdeopaiiitb/unet-checkpoints-v16/checkpoints/unet_best.pt'

# Load best model
best_model = UNet(num_classes=NUM_CLASSES).to(DEVICE)

best_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=DEVICE)
)

best_model.eval()

per_class_iou  = {i: [] for i in range(1, NUM_CLASSES)}
per_class_dice = {i: [] for i in range(1, NUM_CLASSES)}

print("Running full validation...")

with torch.no_grad():
    for imgs, masks in tqdm(full_val_loader, desc='Full Evaluation'):
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = best_model(imgs)
        preds  = logits.argmax(dim=1)

        for cls in range(1, NUM_CLASSES):
            pred_cls   = (preds == cls).float()
            target_cls = (masks == cls).float()

            smooth = 1e-6
            inter  = (pred_cls * target_cls).sum().item()
            union  = (pred_cls + target_cls).clamp(0, 1).sum().item()

            iou  = (inter + smooth) / (union + smooth)
            dice = (2 * inter + smooth) / (
                pred_cls.sum().item() + target_cls.sum().item() + smooth
            )

            per_class_iou[cls].append(iou)
            per_class_dice[cls].append(dice)


print("\nFinal Per-class Results (U-Net — Full Validation):")
print(f"{'Class':<20} {'mIoU':>10} {'Dice':>10}")
print("-" * 42)

all_ious, all_dices = [], []

for cls in range(1, NUM_CLASSES):
    name = IDX_TO_NAME[cls]

    miou = np.mean(per_class_iou[cls])
    dice = np.mean(per_class_dice[cls])

    all_ious.append(miou)
    all_dices.append(dice)

    print(f"{name:<20} {miou:>10.4f} {dice:>10.4f}")

print(f"\nMacro mIoU (foreground) : {np.mean(all_ious):.4f}")
print(f"Macro Dice (foreground) : {np.mean(all_dices):.4f}")

Running full validation...


Full Evaluation: 100%|██████████| 1005/1005 [08:50<00:00,  1.89it/s]


Final Per-class Results (U-Net — Full Validation):
Class                      mIoU       Dice
------------------------------------------
short_sleeve_top         0.5290     0.6359
trousers                 0.5474     0.6548
shorts                   0.3210     0.3919
long_sleeve_top          0.3455     0.4334
skirt                    0.3412     0.4306

Macro mIoU (foreground) : 0.4168
Macro Dice (foreground) : 0.5093


In [33]:
# Upload dataset to kaggle

# import subprocess

# Save training history
pd.DataFrame(history).to_csv('/kaggle/working/unet_history.csv', index=False)

# Create upload directory
os.makedirs('/kaggle/working/unet_upload/', exist_ok=True)

# Copy model + history
shutil.copy(SAVE_DIR + 'unet_best.pt',
            '/kaggle/working/unet_upload/unet_best.pt')

shutil.copy('/kaggle/working/unet_history.csv',
            '/kaggle/working/unet_upload/unet_history.csv')

# Save label mapping
label_mapping = {
    "background"      : 0,
    "short_sleeve_top": 1,
    "trousers"        : 2,
    "shorts"          : 3,
    "long_sleeve_top" : 4,
    "skirt"           : 5,
    "note"            : "0=background (pixels not covered by any polygon)"
}

with open('/kaggle/working/unet_upload/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

# Metadata
metadata = {
    "title"   : "vr-unet-model",
    "id"      : "pankajdeopaiiitb/vr-unet-model",
    "licenses": [{"name": "CC0-1.0"}]
}

with open('/kaggle/working/unet_upload/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f)

# Upload dataset
result = subprocess.run(
    ['kaggle', 'datasets', 'version', '-p',
     '/kaggle/working/unet_upload/',
     '-m', 'update model', '--dir-mode', 'zip'],
    capture_output=True, text=True
)

# If dataset doesn't exist → create
if result.returncode != 0:
    subprocess.run(
        ['kaggle', 'datasets', 'create', '-p',
         '/kaggle/working/unet_upload/', '--dir-mode', 'zip'],
        capture_output=True, text=True
    )
print(result.stdout)
print(result.stderr)

Starting upload for file unet_best.pt
Upload successful: unet_best.pt (93MB)
Starting upload for file label_mapping.json
Upload successful: label_mapping.json (178B)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopaiiitb/vr-unet-model


  0%|          | 0.00/93.1M [00:00<?, ?B/s]
 12%|█▏        | 11.4M/93.1M [00:00<00:00, 99.1MB/s]
 34%|███▍      | 31.9M/93.1M [00:00<00:00, 162MB/s] 
 51%|█████▏    | 47.9M/93.1M [00:00<00:00, 140MB/s]
 66%|██████▋   | 61.8M/93.1M [00:00<00:00, 109MB/s]
 79%|███████▊  | 73.2M/93.1M [00:00<00:00, 107MB/s]
 98%|█████████▊| 91.0M/93.1M [00:00<00:00, 114MB/s]
100%|██████████| 93.1M/93.1M [00:01<00:00, 74.1MB/s]

  0%|          | 0.00/178 [00:00<?, ?B/s]
100%|██████████| 178/178 [00:00<00:00, 523B/s]

